In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate
insurance_data_path = 'insurance.csv'
insurance = pd.read_csv(insurance_data_path)
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,Southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


In [ ]:
for col in insurance.columns:
    print(f"Column: {col}")
    print(insurance[col].value_counts())
    print("-" * 40)


Column: age
 18.0    63
 19.0    62
 48.0    29
 46.0    28
 53.0    27
         ..
-28.0     1
-25.0     1
-56.0     1
-58.0     1
-33.0     1
Name: age, Length: 80, dtype: int64
----------------------------------------
Column: sex
male      517
female    503
man        64
M          64
woman      62
F          62
Name: sex, dtype: int64
----------------------------------------
Column: bmi
32.300    13
28.310     9
31.350     8
34.100     8
28.880     8
          ..
29.100     1
34.295     1
23.465     1
45.430     1
30.970     1
Name: bmi, Length: 539, dtype: int64
----------------------------------------
Column: children
 0.0    551
 1.0    292
 2.0    209
 3.0    140
-1.0     20
 4.0     20
 5.0     16
-2.0     12
-3.0      9
-4.0      3
Name: children, dtype: int64
----------------------------------------
Column: smoker
no     1013
yes     259
Name: smoker, dtype: int64
----------------------------------------
Column: region
Southeast    172
southeast    170
southwest    164
North

In [107]:
insurance["sex"] = (
    insurance["sex"]
    .str.lower()         
    .replace({
        "man": "male",
        "m": "male",
        "woman": "female",
        "f": "female"
    })
)

insurance["region"] = insurance["region"].str.lower()


In [108]:
insurance["sex"].value_counts()


male      645
female    627
Name: sex, dtype: int64

In [109]:
insurance["region"].value_counts()


southeast    342
southwest    312
northwest    310
northeast    308
Name: region, dtype: int64

In [110]:
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


In [111]:
insurance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1272 non-null   float64
 1   sex       1272 non-null   object 
 2   bmi       1272 non-null   float64
 3   children  1272 non-null   float64
 4   smoker    1272 non-null   object 
 5   region    1272 non-null   object 
 6   charges   1284 non-null   object 
dtypes: float64(3), object(4)
memory usage: 73.3+ KB


In [112]:
insurance.describe()

,age,bmi,children
count,1272.000000,1272.000000,1272.000000
mean,35.214623,30.560550,0.948899
std,22.478251,6.095573,1.303532
min,-64.000000,15.960000,-4.000000
25%,24.750000,26.180000,0.000000
50%,38.000000,30.210000,1.000000
75%,51.000000,34.485000,2.000000
max,64.000000,53.130000,5.000000


In [113]:
insurance.isnull().sum()

age         66
sex         66
bmi         66
children    66
smoker      66
region      66
charges     54
dtype: int64

In [114]:
insurance = insurance.dropna()
insurance.isna().sum()


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [115]:
insurance.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1208 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1208 non-null   float64
 1   sex       1208 non-null   object 
 2   bmi       1208 non-null   float64
 3   children  1208 non-null   float64
 4   smoker    1208 non-null   object 
 5   region    1208 non-null   object 
 6   charges   1208 non-null   object 
dtypes: float64(3), object(4)
memory usage: 75.5+ KB


In [ ]:
insurance['charges'] = insurance['charges'].replace({'\$': ''}, regex=True).astype(float)

insurance = insurance[insurance['age'] > 0]

insurance.loc[insurance['children'] < 0, 'children'] = 0


In [117]:
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.92400
1,18.0,male,33.770,1.0,no,southeast,1725.55230
2,28.0,male,33.000,3.0,no,southeast,4449.46200
3,33.0,male,22.705,0.0,no,northwest,21984.47061
4,32.0,male,28.880,0.0,no,northwest,3866.85520


In [118]:
X = insurance[["age", "bmi", "children", "sex", "smoker", "region"]]
y = insurance["charges"]

In [119]:
X_encoded = pd.get_dummies(
    X,
    columns=["sex", "smoker", "region"],
    drop_first=True
)

In [120]:
num_cols = ["age", "bmi", "children"]

scaler = StandardScaler()
X_encoded[num_cols] = scaler.fit_transform(X_encoded[num_cols])

In [121]:
X_encoded.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,-1.427171,-0.439874,-0.853770,0,1,0,0,1
1,-1.497807,0.519066,-0.014607,1,0,0,1,0
2,-0.791445,0.393276,1.663719,1,0,0,1,0
3,-0.438264,-1.288543,-0.853770,1,0,1,0,0
4,-0.508900,-0.279778,-0.853770,1,0,1,0,0


In [ ]:
insurance_model = LinearRegression()

r2_scores = cross_val_score(
    insurance_model,
    X_encoded,
    y,
    cv=5,
    scoring="r2"
)

insurance_model.fit(X_encoded, y)


r2_score = r2_scores.mean()

r2_score

0.7450511466263761

In [123]:
validation_data = pd.read_csv("validation_dataset.csv")
validation_data.head()

,age,sex,bmi,children,smoker,region
0,18.0,female,24.090000,1.0,no,southeast
1,39.0,male,26.410000,0.0,yes,northeast
2,27.0,male,29.150000,0.0,yes,southeast
3,71.0,male,65.502135,13.0,yes,southeast
4,28.0,male,38.060000,0.0,no,southeast


In [124]:
validation_data.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
dtype: int64

In [125]:
validation_data["sex"] = (
    validation_data["sex"]
    .str.lower()
    .replace({
        "man": "male",
        "m": "male",
        "woman": "female",
        "f": "female"
    })
)
validation_data["region"] = validation_data["region"].str.lower()
validation_data = validation_data[validation_data["age"] > 0]
validation_data.loc[validation_data["children"] < 0, "children"] = 0
validation_data.head()

,age,sex,bmi,children,smoker,region
0,18.0,female,24.090000,1.0,no,southeast
1,39.0,male,26.410000,0.0,yes,northeast
2,27.0,male,29.150000,0.0,yes,southeast
3,71.0,male,65.502135,13.0,yes,southeast
4,28.0,male,38.060000,0.0,no,southeast


In [126]:
validation_data = pd.get_dummies(
    validation_data,
    columns=["sex", "smoker", "region"],
    drop_first=True
)

num_cols = ["age", "bmi", "children"]
validation_data[num_cols] = scaler.transform(validation_data[num_cols])


In [127]:
validation_data.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,-1.497807,-1.062286,-0.014607,0,0,0,1,0
1,-0.014447,-0.683284,-0.853770,1,1,0,0,0
2,-0.862081,-0.235670,-0.853770,1,1,0,1,0
3,2.245911,5.702914,10.055348,1,1,0,1,0
4,-0.791445,1.219892,-0.853770,1,0,0,1,0


In [ ]:

if "predicted_charges" in validation_data.columns:
    validation_data = validation_data.drop(columns=["predicted_charges"])

validation_data["predicted_charges"] = insurance_model.predict(validation_data)
validation_data.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,predicted_charges
0,-1.497807,-1.062286,-0.014607,0,0,0,1,0,508.145323
1,-0.014447,-0.683284,-0.853770,1,1,0,0,0,30947.521922
2,-0.862081,-0.235670,-0.853770,1,1,0,1,0,27951.157717
3,2.245911,5.702914,10.055348,1,1,0,1,0,56291.274683
4,-0.791445,1.219892,-0.853770,1,0,0,1,0,7147.814884


In [ ]:
validation_data.loc[
    validation_data["predicted_charges"] < 1000,
    "predicted_charges"
] = 1000
validation_data.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,predicted_charges
0,-1.497807,-1.062286,-0.014607,0,0,0,1,0,1000.000000
1,-0.014447,-0.683284,-0.853770,1,1,0,0,0,30947.521922
2,-0.862081,-0.235670,-0.853770,1,1,0,1,0,27951.157717
3,2.245911,5.702914,10.055348,1,1,0,1,0,56291.274683
4,-0.791445,1.219892,-0.853770,1,0,0,1,0,7147.814884
